In [ ]:
import pickle
from typing import Literal
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.tqdm
import pandas as pd
import numpy as np
import plotly.express as px

from python.config import load_config

cfg = load_config(os.environ.get('CONFIG', 'config.json'))

import python.plots as plots
from python.error_metrics import LoadedErrorMetrics, FilteredErrorMetrics, MetricsIndex as MI # Alias
from python.sampled_hierarchies import SampledHierarchies, SingleStepHierarchies, PartitionStepHierarchies
from python.statistical_test import slope_ttest, page_trend_test, CombineMethod, wilcoxon_test, TrendType, sign_test, per_series_test
from python.aggregation import Aggregation

from itertools import product

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [ ]:
AVAILABLE_LEVELS = Aggregation.complete_summing_matrix(n=cfg.LEVELS.SUBSET, seed=42)

if cfg.is_single_step_hierarchies():
    if cfg.LEVELS.PARTITION == 0:
        sh = SingleStepHierarchies(AVAILABLE_LEVELS)
    else:
        sh = PartitionStepHierarchies(AVAILABLE_LEVELS, cfg.LEVELS.PARTITION, cfg.LEVELS.BASE_PERIOD)
else:
    sh = SampledHierarchies(AVAILABLE_LEVELS, cfg.LEVELS.DIFF, cfg.LEVELS.SEQUENCES, max_h=cfg.LEVELS.MAX_H)

datasets = ['web-traffic', 'M4', 'M5']
models = ['AutoETS', 'AutoARIMA', 'Prophet']

ems = {}

for d, m in product(datasets, models):
    ID = f"{d}/{m}"
    em = FilteredErrorMetrics(LoadedErrorMetrics(f"metrics/{cfg.MASTER_FOLDER}/{ID}/sample_hiers_errors.csv"), MI(error_metric='nrmse'))
    ems[ID] = em

In [ ]:
comprehensive_dfs = []
NOISES = (0.1, 0.25, 0.5, 0.75, 1)

for d in datasets:
    for m in models:
        ID = f"{d}/{m}"
        em = ems[ID]
        df = pd.DataFrame(index=em.ts_columns)
        df.index.name = 'series_id'

        df['data'] = ID

        df['orig_bottom'] = em.get_error(MI(hierarchy='Minimal', forecast_type='reconciled_forecasts', granularity='Bottom')).T
        df['orig_top']    = em.get_error(MI(hierarchy='Minimal', forecast_type='reconciled_forecasts', granularity='Top')).T

        df['new_bot_mean'] = em.get_error(MI(hierarchy=sh.hierarchies, forecast_type='reconciled_forecasts', granularity='Bottom')).mean(axis=0).values
        df['new_top_mean'] = em.get_error(MI(hierarchy=sh.hierarchies, forecast_type='reconciled_forecasts', granularity='Top')).mean(axis=0).values

        df['new_bot_median'] = em.get_error(MI(hierarchy=sh.hierarchies, forecast_type='reconciled_forecasts', granularity='Bottom')).median(axis=0).values
        df['new_top_median'] = em.get_error(MI(hierarchy=sh.hierarchies, forecast_type='reconciled_forecasts', granularity='Top')).median(axis=0).values

        top_base = em.get_error(MI(hierarchy='Minimal', forecast_type='reconciled_forecasts', granularity='Top')).values.flatten()
        bot_base = em.get_error(MI(hierarchy='Minimal', forecast_type='reconciled_forecasts', granularity='Bottom')).values.flatten()

        bot_diffs = em.get_error(MI(hierarchy=sh.hierarchies, forecast_type='reconciled_forecasts', granularity='Bottom')).values - bot_base
        top_diffs = em.get_error(MI(hierarchy=sh.hierarchies, forecast_type='reconciled_forecasts', granularity='Top')).values - top_base


        def percentage_true(array) -> float:
            return (array).sum(axis=0) / len(sh.hierarchies)

        def for_each_noise(base_at_granularity):
            for noise in NOISES:
                threshold = base_at_granularity * noise / 100
                yield noise, threshold

        df['bot', 'le0_%'] = percentage_true(bot_diffs <= 0)
        df['bot', 'gt0_%'] = 1 - df['bot', 'le0_%']

        for noise, threshold in for_each_noise(bot_base):
            df['bot', f'noise{noise}%_%'] = percentage_true((bot_diffs <= threshold) & (bot_diffs >= -threshold))
        for noise, threshold in for_each_noise(bot_base):
            df['bot', f'gt{noise}%_%'] = percentage_true(bot_diffs > threshold)


        df['top', 'le0_%'] = percentage_true(top_diffs <= 0)
        df['top', 'gt0_%'] = 1 - df['top', 'le0_%']

        for noise, threshold in for_each_noise(top_base):
            df['top', f'noise{noise}%_%'] = percentage_true((top_diffs <= threshold) & (top_diffs >= -threshold))
        for noise, threshold in for_each_noise(top_base):
            df['top', f'gt{noise}%_%'] = percentage_true(top_diffs > threshold)

        comprehensive_dfs.append(df)

comprehensive_df = pd.concat(comprehensive_dfs, axis=0)
comprehensive_df.columns = pd.MultiIndex.from_tuples([c if isinstance(c, tuple) else (c, "") for c in comprehensive_df.columns])

In [ ]:
comprehensive_df

In [ ]:
comprehensive_df.describe()

In [ ]:
noise_perc = 1
granularity = 'bot'

vals = pd.DataFrame()
vals['percentages'] = comprehensive_df[granularity, f'gt{noise_perc}%_%'].values
vals['group'] = vals['percentages'].apply(lambda x: 'Below median' if x <= vals['percentages'].median() else 'Above median')

px.histogram(vals, x='percentages', color='group', labels={'percentages': f'fractions where increase was greater than {noise_perc}% noise'})

In [ ]:
granularity = 'bot'
stat = 'gt0_%'

vals = pd.DataFrame()
vals['percentages'] = comprehensive_df[granularity, stat].values
vals['group'] = vals['percentages'].apply(lambda x: 'Below median' if x <= vals['percentages'].median() else 'Above median')

px.histogram(vals, x='percentages', color='group', labels={'percentages': f'Fractions where {stat}'})